# 03 Pawn advancement

Visualisations based on the "pawn advancement" metric(s)

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import polars as pl
import altair as alt

import pawncounter
import pawncounter.util as pu

In [61]:
BOARD_WIDTH = 3
BOARD_DEPTH = 4
pc = pawncounter.PawnCounter(BOARD_WIDTH, BOARD_DEPTH)

In [62]:
transitions = pc.generate_transitions()
init_pos = pc.init_position()

In [68]:
positions = pc.extract_positions(transitions)

_pos = pl.col("pos")
positions = positions.with_columns(
    white_pawn_count=(_pos & pu.white_mask()).bitwise_count_ones(),
    black_pawn_count=(_pos & pu.black_mask()).bitwise_count_ones(),
).with_columns(
    white_advancement=sum(
        _pos.and_(pu.white_mask()).and_(pu.rank_mask(r)).bitwise_count_ones() * r
        for r in range(pc.depth)
    )
    + (pc.width - pl.col("white_pawn_count")) * pc.depth,
    black_advancement=sum(
        _pos.and_(pu.black_mask()).and_(pu.rank_mask(r)).bitwise_count_ones()
        * (pc.depth - r - 1)
        for r in range(pc.depth)
    )
    + (pc.width - pl.col("black_pawn_count")) * pc.depth,
)
positions

position_id,pos,white_pawn_count,black_pawn_count,white_advancement,black_advancement
u32,u128,u32,u32,u32,u32
0,9709333062732580235837697,3,3,0,0
1,9709333062732580235837698,3,3,1,0
2,9709333062732580235838208,3,3,1,0
3,9709333062732580235837696,2,3,4,0
4,9709333062732580235837953,3,3,1,0
…,…,…,…,…,…
42775,1213666632841572529996808,3,3,8,9
42776,1213666632841572530257928,3,3,8,9
42777,1213666632841572529995788,3,3,8,9


In [90]:
advancements_heatmap = (
    alt.Chart(
        positions.group_by("white_advancement", "black_advancement")
        .agg(position_count=pl.len())
        .sort("white_advancement", "black_advancement")
    )
    .mark_rect()
    .encode(
        alt.X("white_advancement:O"),
        alt.Y("black_advancement:O").scale(reverse=True),
        alt.Color("position_count:Q").scale(scheme="bluegreen"),
    )
)
advancements_heatmap

alt.Chart(...)

In [126]:
(
    alt.Chart(
        transitions.lazy()
        .join(
            positions.lazy().select(
                start_pos="pos",
                start_wav="white_advancement",
                start_bav="black_advancement",
            ),
            on="start_pos",
        )
        .join(
            positions.lazy().select(
                end_pos="pos",
                end_wav="white_advancement",
                end_bav="black_advancement",
            ),
            on="end_pos",
        )
        .group_by("start_wav", "start_bav", "end_wav", "end_bav")
        .agg(transition_count=pl.len())
        .sort("start_wav", "start_bav", "end_wav", "end_bav")
        .collect()
    )
    .mark_rule(opacity=0.7)
    .encode(
        alt.X("start_wav:O"),
        alt.Y("start_bav:O").scale(reverse=True),
        alt.X2("end_wav:O"),
        alt.Y2("end_bav:O"),
        alt.StrokeWidth("transition_count:Q").scale(
            domain=[0, 6000],
            range=[0.5, 10],
            # type="log",
        ),
        alt.Color("transition_count:Q").scale(scheme="lightgreyred"),
    )
    .properties(width=700, height=700)
)

alt.Chart(...)

In [124]:
(
    alt.Chart(
        transitions.lazy()
        .join(
            positions.lazy().select(
                start_pos="pos",
                start_wc="white_pawn_count",
                start_bc="black_pawn_count",
            ),
            on="start_pos",
        )
        .join(
            positions.lazy().select(
                end_pos="pos",
                end_wc="white_pawn_count",
                end_bc="black_pawn_count",
            ),
            on="end_pos",
        )
        .group_by("start_wc", "start_bc", "end_wc", "end_bc")
        .agg(transition_count=pl.len())
        .sort("start_wc", "start_bc", "end_wc", "end_bc")
        .collect()
    )
    .mark_rule()
    .encode(
        alt.X("start_wc:O").scale(reverse=True),
        alt.Y("start_bc:O"),
        alt.X2("end_wc:O"),
        alt.Y2("end_bc:O"),
        # alt.StrokeWidth("transition_count:Q").scale(domain=[0, 6000], range=[0.1, 1]),
        alt.Color("transition_count:Q").scale(scheme="lightgreyred"),
    )
    .properties(width=200, height=200)
)

alt.Chart(...)